# ProxyLens 04: export a trained checkpoint on CPU (no GPU needed)

**When:** like notebook 03, but for when Colab offers no GPU (free T4 quota used up). Training is
already done; merging the LoRA, converting to GGUF and quantizing all run on CPU. Use
*Runtime > Change runtime type > CPU*.

**How:** plain `transformers` + `peft` merge the adapter into the full-precision
`Qwen2.5-3B-Instruct` (bf16, ~6.2 GB of the ~12 GB RAM), then llama.cpp's own
`convert_hf_to_gguf.py` and `llama-quantize` produce `Q4_K_M`. This is what Unsloth's
`save_pretrained_gguf` does in notebooks 01/03.

**Inputs:** the Drive folder notebook 01 wrote (`MyDrive/proxylens/qwen3b-lora-v1/checkpoint-*`);
Colab secret `HF_TOKEN` with write access.

**Outputs:** LoRA adapter + `metrics.json` at `<you>/proxylens-qwen3b-lora-v1` and
`proxylens-q4_k_m.gguf` at `<you>/proxylens-qwen3b-gguf-v1`.

**Runtime:** about 30-45 min on the free CPU runtime (building llama-quantize and quantizing
take the longest).

In [ ]:
%%capture
!pip install -q "peft>=0.13" "huggingface_hub>=0.25" "pydantic>=2.8" sentencepiece
# Colab preinstalls torchao 0.10, which newer peft refuses to import; the merge does not use it.
!pip uninstall -q -y torchao
# llama.cpp: the HF -> GGUF converter, and the quantizer built from source (CPU only).
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!cmake -S llama.cpp -B llama.cpp/build -DGGML_NATIVE=OFF -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF
!cmake --build llama.cpp/build --target llama-quantize -j 4
# Prebuilt CPU wheel, for the sanity check on the finished GGUF.
!pip install -q llama-cpp-python --prefer-binary --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

In [ ]:
import gc, glob, json, os, shutil, subprocess
import torch
from google.colab import drive, userdata
from huggingface_hub import HfApi, snapshot_download

QUANTIZE = "llama.cpp/build/bin/llama-quantize"
assert os.path.isfile(QUANTIZE), "llama-quantize did not build; re-run the install cell without %%capture"
print("torch", torch.__version__, "| CPU threads:", os.cpu_count())

HF_TOKEN = userdata.get("HF_TOKEN")  # Colab secret; needs write access
HF_USER = HfApi(token=HF_TOKEN).whoami()["name"]
DATASET_REPO = f"{HF_USER}/proxylens-sft-v1"
LORA_REPO = f"{HF_USER}/proxylens-qwen3b-lora-v1"
GGUF_REPO = f"{HF_USER}/proxylens-qwen3b-gguf-v1"
GGUF_FILE = "proxylens-q4_k_m.gguf"                     # = STUDENT_GGUF_FILE in .env
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"     # what the adapter was trained on
MERGE_BASE = "unsloth/Qwen2.5-3B-Instruct"              # same weights, full precision
print("HF user:", HF_USER)

In [ ]:
drive.mount("/content/drive")
OUTPUT_DIR = "/content/drive/MyDrive/proxylens/qwen3b-lora-v1"
CHECKPOINT = None   # e.g. f"{OUTPUT_DIR}/checkpoint-130" to choose one; None = latest complete

def complete(path):
    # Finished syncing to Drive: adapter weights + trainer state present.
    weights = os.path.join(path, "adapter_model.safetensors")
    return (os.path.isfile(weights) and os.path.getsize(weights) > 50_000_000
            and os.path.isfile(os.path.join(path, "trainer_state.json")))

found = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
for path in found:
    print(f"{os.path.basename(path):16} {'complete' if complete(path) else 'INCOMPLETE (skipped)'}")
if CHECKPOINT is None:
    usable = [p for p in found if complete(p)]
    assert usable, f"no complete checkpoint in {OUTPUT_DIR}"
    CHECKPOINT = usable[-1]
print("using", CHECKPOINT)

## Merge the adapter into the full-precision base

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = AutoModelForCausalLM.from_pretrained(MERGE_BASE, torch_dtype=torch.bfloat16,
                                            low_cpu_mem_usage=True, token=HF_TOKEN)
merged = PeftModel.from_pretrained(base, CHECKPOINT).merge_and_unload()
has_tokenizer = os.path.isfile(f"{CHECKPOINT}/tokenizer_config.json")
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT if has_tokenizer else MERGE_BASE)
merged.save_pretrained("merged", safe_serialization=True)
tokenizer.save_pretrained("merged")
del base, merged
gc.collect()
print(sorted(os.listdir("merged")))

## Convert to GGUF and quantize to Q4_K_M

In [ ]:
def run(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

run(["python", "llama.cpp/convert_hf_to_gguf.py", "merged", "--outfile", "proxylens-f16.gguf",
     "--outtype", "f16"])
shutil.rmtree("merged")   # free disk before quantizing
run([QUANTIZE, "proxylens-f16.gguf", GGUF_FILE, "Q4_K_M", str(os.cpu_count())])
os.remove("proxylens-f16.gguf")
print(GGUF_FILE, round(os.path.getsize(GGUF_FILE) / 1e9, 2), "GB")

In [ ]:
# Copied from app/schemas/resolution.py by scripts/build_notebooks.py. Do not edit here.
"""Resolution extraction schema (SPEC 5.1). Also the fine-tuning target format."""

from enum import StrEnum

from pydantic import BaseModel, Field


class ResolutionType(StrEnum):
    ADOPT_FINANCIALS = "ADOPT_FINANCIALS"
    DIVIDEND = "DIVIDEND"
    DIRECTOR_REAPPOINT_ROTATION = "DIRECTOR_REAPPOINT_ROTATION"
    INDEPENDENT_DIRECTOR_APPOINT = "INDEPENDENT_DIRECTOR_APPOINT"
    NON_INDEPENDENT_DIRECTOR_APPOINT = "NON_INDEPENDENT_DIRECTOR_APPOINT"
    MANAGERIAL_REMUNERATION = "MANAGERIAL_REMUNERATION"
    RELATED_PARTY_TRANSACTION = "RELATED_PARTY_TRANSACTION"
    AUDITOR_APPOINT = "AUDITOR_APPOINT"
    AUDITOR_REMUNERATION_COST = "AUDITOR_REMUNERATION_COST"  # cost auditor ratification
    ESOP = "ESOP"
    CAPITAL_RAISE = "CAPITAL_RAISE"  # QIP, preferential, NCDs
    BORROWING_LIMITS = "BORROWING_LIMITS"  # Sec 180
    ARTICLES_OR_MOA_AMENDMENT = "ARTICLES_OR_MOA_AMENDMENT"
    OTHER = "OTHER"


class Person(BaseModel):
    name: str
    role: str | None = None
    age: int | None = None
    din: str | None = None
    is_promoter: bool | None = None
    tenure_years_proposed: float | None = None
    prior_tenure_years: float | None = None
    board_attendance_pct: float | None = None


class Money(BaseModel):
    amount_inr: float | None = None  # normalized to INR (crore/lakh -> INR)
    raw: str


class ResolutionExtraction(BaseModel):
    item_no: int
    title: str
    resolution_type: ResolutionType
    is_special_resolution: bool
    persons: list[Person] = Field(default_factory=list)
    amounts: list[Money] = Field(default_factory=list)
    counterparty: str | None = None
    counterparty_is_related: bool | None = None
    transaction_nature: str | None = None
    duration_years: float | None = None
    key_facts: list[str] = Field(default_factory=list, max_length=5)

In [ ]:
# Sanity check before uploading: the quantized model must produce valid extractions.
# Unconstrained decoding, so this measures the model, not the JSON grammar. ~1-2 min per item.
from llama_cpp import Llama

data_dir = snapshot_download(DATASET_REPO, repo_type="dataset", token=HF_TOKEN,
                             allow_patterns=["sft/val.jsonl"])
val = [json.loads(line) for line in open(f"{data_dir}/sft/val.jsonl")][:3]
llm = Llama(model_path=GGUF_FILE, n_ctx=4096, n_threads=os.cpu_count(), verbose=False)
ok = 0
for row in val:
    out = llm.create_chat_completion(messages=row["messages"][:2], temperature=0.0, max_tokens=1024)
    text = out["choices"][0]["message"]["content"]
    try:
        pred = ResolutionExtraction.model_validate_json(text)
        ok += 1
        print("valid", pred.resolution_type.value)
    except Exception as exc:
        print("INVALID:", str(exc)[:200])
del llm
assert ok >= 2, "quantized model does not look trained; check CHECKPOINT"
SANITY = f"{ok}/3 valid (Q4_K_M, CPU)"

In [ ]:
# metrics.json from the trainer state saved with the checkpoint.
state = json.load(open(f"{CHECKPOINT}/trainer_state.json"))
metrics = {
    "base_model": BASE_MODEL,
    "checkpoint": os.path.basename(CHECKPOINT),
    "epochs": state.get("epoch"),
    "global_step": state.get("global_step"),
    "train_loss_log": [h for h in state["log_history"] if "loss" in h],
    "eval_loss_log": [h for h in state["log_history"] if "eval_loss" in h],
    "sanity_check": SANITY,
}
with open(f"{OUTPUT_DIR}/metrics.json", "w") as fh:
    json.dump(metrics, fh, indent=2)
print(json.dumps(metrics["eval_loss_log"], indent=2))

## Upload the LoRA adapter and the GGUF (private HF repos)

In [ ]:
api = HfApi(token=HF_TOKEN)
api.create_repo(LORA_REPO, private=True, exist_ok=True)
# Adapter + tokenizer only; optimizer/scheduler/RNG state stays on Drive.
api.upload_folder(folder_path=CHECKPOINT, repo_id=LORA_REPO,
                  allow_patterns=["adapter_*", "*.json", "*.jinja", "merges.txt"],
                  ignore_patterns=["trainer_state.json", "training_args*", "rng_state*"])
api.upload_file(path_or_fileobj=f"{OUTPUT_DIR}/metrics.json", path_in_repo="metrics.json",
                repo_id=LORA_REPO)

api.create_repo(GGUF_REPO, private=True, exist_ok=True)
api.upload_file(path_or_fileobj=GGUF_FILE, path_in_repo=GGUF_FILE, repo_id=GGUF_REPO)
print(f"Uploaded https://huggingface.co/{GGUF_REPO}/blob/main/{GGUF_FILE}")

## Next

In the repo's `.env` set `STUDENT_GGUF_REPO=<you>/proxylens-qwen3b-gguf-v1` (and `HF_TOKEN`), then run
the Phase 4 smoke test:

```bash
uv sync --extra student
uv run python -m scripts.smoke_student --n 10
```